# Preprocessing pipeline review (Google Colab / Drive)

Top-to-bottom review of **NIfTI / DICOM → nnUNet-ready labels**, focused on:

1. **Background vs `other-tissue`** quality
2. **Stable visualisation** (same organ → same colour)
3. Whether **bad inputs** explain low GTVp Dice (vs nnUNet itself)

Audit set: **4 RADCURE** + **4 HECKTOR** cases.

| nnUNet stem | Original ID (used for download) |
|-------------|-------------------------------|
| `case_0122` | `RADCURE-0122` |
| `case_0040` | `RADCURE-0040` |
| `case_0397` | `RADCURE-0397` |
| `case_0151` | `RADCURE-0151` |
| `case_012` | `HMR-012` *(provisional center; falls back to any matching number)* |
| `case_023` | `CHUM-023` *(provisional)* |
| `case_098` | `CHUM-098` *(provisional)* |
| `case_057` | `HMR-057` *(provisional)* |

**Colab tip:** after the install cell → **Runtime → Restart session** → remount Drive → run the import-only cell (skip pip).

**Handoff / learnings:** see [`FINDINGS.md`](FINDINGS.md) in this folder (Steps A–B done; Steps C–E = TotalSegmentator + fixed organ dict + tumor viz).


## 0. Colab setup — Drive, repo, package


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# Paths: prefer an existing clone on Drive; otherwise clone from GitHub.
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
WORK_DIR = DRIVE_ROOT / "preprocessing_pipeline_review"
WORK_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
GITHUB_REPO_URL = "https://github.com/xiscapericas/my_tailors_drawer.git"
CLONE_DIR = Path("/content/my_tailors_drawer")
REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")

if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
    REPO_ROOT = DRIVE_REPO
    print("Using Drive repo:", REPO_ROOT)
else:
    if not (CLONE_DIR / REPO_SUBPATH / "setup.py").is_file():
        !git clone --depth 1 {GITHUB_REPO_URL} {CLONE_DIR}
    REPO_ROOT = CLONE_DIR / REPO_SUBPATH
    print("Using cloned repo:", REPO_ROOT)

assert (REPO_ROOT / "setup.py").is_file(), f"setup.py not found under {REPO_ROOT}"
assert (REPO_ROOT / "image_processor").is_dir()
os.chdir(REPO_ROOT)
print("cwd:", Path.cwd())


In [ ]:
# Install — Step A deps only (NO TotalSegmentator yet).
# MUST install from REPO_ROOT (not /content).
# After this cell: Runtime → Restart session → remount Drive → import-only cell.
#
# NumPy: use >=2.1 (Colab imagecodecs requires it). Do NOT force-reinstall 2.0.2.
# Pip "ERROR: dependency resolver" lines for datasets/dill/cucim are usually
# warnings about preinstalled Colab packages — OK if imports succeed after restart.

import os
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
CLONE_DIR = Path("/content/my_tailors_drawer")
REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")

if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
    REPO_ROOT = DRIVE_REPO
elif (CLONE_DIR / REPO_SUBPATH / "setup.py").is_file():
    REPO_ROOT = CLONE_DIR / REPO_SUBPATH
else:
    raise FileNotFoundError(
        "Cannot find radcure-medical-imaging (setup.py missing). "
        "Re-run Drive mount + clone/path cell first."
    )

os.chdir(REPO_ROOT)
print("Installing from:", REPO_ROOT)

pkgs = [
    "numpy>=2.1,<2.3",
    "boto3",
    "python-dotenv",
    "blosc2>=2.5.0",
    "nibabel",
    "SimpleITK",
    "pydicom",
    "rt-utils",
    "matplotlib",
    "scikit-image>=0.19.0,<0.26.0",
    "scipy",
    "opencv-python-headless",
    "tqdm",
    "p-tqdm",
    "seaborn",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs],
    cwd=str(REPO_ROOT),
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)],
    cwd=str(REPO_ROOT),
)
# Keep NumPy in the 2.1–2.2 band last (without downgrading to 2.0.x)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "numpy>=2.1,<2.3"],
    cwd=str(REPO_ROOT),
)

print("Install done from", REPO_ROOT)
print(">>> Runtime → Restart session")
print(">>> Then: remount Drive → run NEXT cell (skip this pip cell)")


In [ ]:
# Run AFTER Runtime → Restart session.
# Re-run Drive mount if needed, then this cell. Skip pip.

import os
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
WORK_DIR = DRIVE_ROOT / "preprocessing_pipeline_review"
WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
CLONE_DIR = Path("/content/my_tailors_drawer")
REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")

if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
    REPO_ROOT = DRIVE_REPO
else:
    REPO_ROOT = CLONE_DIR / REPO_SUBPATH

assert (REPO_ROOT / "image_processor").is_dir(), f"Repo not found at {REPO_ROOT}"
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import image_processor

print("cwd:", Path.cwd())
print("numpy:", np.__version__)
print("image_processor OK:", image_processor.__file__)


### Step C prep — TotalSegmentator install (run when starting C)

Skip until Steps A–B are done. Then:

```python
!pip install -q totalsegmentator
!pip install -q "numpy>=2.1,<2.3"
```

**Runtime → Restart session** → remount Drive → re-run the **import-only** cell (not the Step-A pip cell).

Ignore pip resolver noise about `datasets` / `dill` / `multiprocess` / `cucim` unless `import numpy` or `import totalsegmentator` actually fails.

Full Step C cells are at the bottom of this notebook.


## 1. Config & AWS credentials

Do **not** hardcode secrets. Prefer Colab Secrets or `getpass`.

Required: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_BUCKET_NAME`, `AWS_FOLDER`.
Optional: `HECKTOR_S3_URI`.


In [ ]:
import os
from getpass import getpass
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
WORK_DIR = DRIVE_ROOT / "preprocessing_pipeline_review"
WORK_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata

    def _secret(name: str, default: str = "") -> str:
        try:
            return userdata.get(name)
        except Exception:
            return default
except ImportError:

    def _secret(name: str, default: str = "") -> str:
        return default


def ensure_env(key: str, prompt: str, secret: bool = False) -> str:
    val = os.environ.get(key) or _secret(key)
    if not val:
        val = getpass(prompt) if secret else input(prompt)
    os.environ[key] = val.strip()
    return os.environ[key]


ensure_env("AWS_ACCESS_KEY_ID", "AWS_ACCESS_KEY_ID: ", secret=True)
ensure_env("AWS_SECRET_ACCESS_KEY", "AWS_SECRET_ACCESS_KEY: ", secret=True)
os.environ.setdefault("AWS_REGION", _secret("AWS_REGION", "eu-west-1") or "eu-west-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

AWS_BUCKET_NAME = ensure_env(
    "AWS_BUCKET_NAME", "AWS_BUCKET_NAME (e.g. xisca-lab): ", secret=False
)
AWS_FOLDER = (
    os.environ.get("AWS_FOLDER")
    or _secret("AWS_FOLDER", "RADCURE/all_cases/")
    or "RADCURE/all_cases/"
)
os.environ["AWS_FOLDER"] = AWS_FOLDER

HECKTOR_S3_URI = (
    os.environ.get("HECKTOR_S3_URI")
    or _secret("HECKTOR_S3_URI", "s3://xisca-lab/HECKTOR/test1.zip")
    or "s3://xisca-lab/HECKTOR/test1.zip"
)
os.environ["HECKTOR_S3_URI"] = HECKTOR_S3_URI

DATA_ROOT = WORK_DIR / "audit_cases"
RADCURE_ROOT = DATA_ROOT / "radcure"
HECKTOR_ROOT = DATA_ROOT / "hecktor"
HECKTOR_DOWNLOAD_DIR = DATA_ROOT / "hecktor_download"
ORGAN_DICT_PATH = WORK_DIR / "audit_organ_dictionary.json"

for p in (RADCURE_ROOT, HECKTOR_ROOT, HECKTOR_DOWNLOAD_DIR):
    p.mkdir(parents=True, exist_ok=True)

# --- Audit cases ---
# RADCURE: case_XXXX -> RADCURE-XXXX
RADCURE_CASE_IDS = ["RADCURE-0122", "RADCURE-0040", "RADCURE-0397", "RADCURE-0151"]

# HECKTOR stems case_012/023/098/057 — center provisional (seed 42).
# Download falls back to any matching *-NNN if preferred ID is missing.
HECKTOR_NNUNET_STEMS = ["case_012", "case_023", "case_098", "case_057"]
HECKTOR_CASE_IDS = ["HMR-012", "CHUM-023", "CHUM-098", "HMR-057"]

SLICE_EXPANSION = 5

print("DATA_ROOT:", DATA_ROOT)
print("RADCURE:", RADCURE_CASE_IDS)
print("HECKTOR stems:", HECKTOR_NNUNET_STEMS)
print("HECKTOR provisional IDs:", HECKTOR_CASE_IDS)
print("AWS:", AWS_BUCKET_NAME, AWS_FOLDER)
print("HECKTOR S3:", HECKTOR_S3_URI)


## 2. Download audit cases

### 2a. RADCURE — 4 case zips from S3

Uses `AWSHandler` + `FileHandler`. Does **not** run TotalSegmentator yet.


In [ ]:
from image_processor.io.aws_handler import AWSHandler
from image_processor.io.file_handler import FileHandler

aws = AWSHandler(
    bucket_name=AWS_BUCKET_NAME,
    aws_folder=AWS_FOLDER,
    region_name=os.environ["AWS_REGION"],
)

radcure_local = {}
for case_id in RADCURE_CASE_IDS:
    case_dir = RADCURE_ROOT / case_id
    zip_path = RADCURE_ROOT / f"{case_id}.zip"
    if not case_dir.is_dir():
        if not zip_path.is_file():
            print(f"Downloading {case_id} ...")
            aws.download_case(case_id, str(RADCURE_ROOT))
        print(f"Unzipping {case_id} ...")
        FileHandler.unzip_file(str(zip_path), str(case_dir))
    else:
        print(f"Already present: {case_dir}")
    radcure_local[case_id] = case_dir

print("RADCURE ready:", {k: str(v) for k, v in radcure_local.items()})


### 2b. HECKTOR — 4 cases

Priority:
1. Reuse under `HECKTOR_ROOT` if ready.
2. Else resolve IDs (exact or any matching number) from Drive or S3 zip, extract only those cases.


In [ ]:
import shutil
import zipfile
from urllib.parse import urlparse

from image_processor.conventions import get_hecktor_paths


def resolve_hecktor_case_id(preferred_id: str, available_ids: list) -> str:
    # Prefer exact ID; else any folder whose numeric suffix matches.
    if preferred_id in available_ids:
        return preferred_id
    num = preferred_id.split("-")[-1]
    matches = [a for a in available_ids if a.split("-")[-1] == num]
    if not matches:
        raise FileNotFoundError(
            f"No HECKTOR case for preferred {preferred_id} (suffix {num}). "
            f"Available sample: {available_ids[:15]}"
        )
    chosen = sorted(matches)[0]
    print(f"  {preferred_id} not found -> using {chosen} (suffix {num})")
    return chosen


def list_hecktor_case_ids_in_zip(zip_path: Path) -> list:
    ids = set()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for n in zf.namelist():
            for part in Path(n).parts:
                if part.endswith(".nii.gz") or part in ("", ".", "__MACOSX"):
                    continue
                if "-" in part:
                    tail = part.split("-")[-1]
                    if tail.isdigit() and 1 <= len(tail) <= 4:
                        ids.add(part)
    return sorted(ids)


def hecktor_case_ready(cases_root: Path, case_id: str) -> bool:
    paths = get_hecktor_paths(str(cases_root / case_id), case_id)
    return os.path.isfile(paths["path_ct"]) and os.path.isfile(paths["path_mask"])


def download_s3_file(s3_uri: str, local_path: Path, region: str = "eu-west-1") -> Path:
    import boto3

    parsed = urlparse(s3_uri)
    bucket, key = parsed.netloc, parsed.path.lstrip("/")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_file():
        print("Zip already on disk:", local_path)
        return local_path
    print(f"Downloading {s3_uri} -> {local_path}")
    boto3.client("s3", region_name=region).download_file(bucket, key, str(local_path))
    return local_path


def extract_hecktor_cases_from_zip(zip_path: Path, case_ids: list, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        for case_id in case_ids:
            members = [
                n
                for n in names
                if f"/{case_id}/" in f"/{n}" or n.startswith(f"{case_id}/")
            ]
            if not members:
                raise FileNotFoundError(f"{case_id} not found inside {zip_path.name}")
            print(f"Extracting {case_id}: {len(members)} files")
            for m in members:
                zf.extract(m, dest_root)

    from pipelines.hecktor.test_pipeline import detect_hecktor_cases_root

    return Path(detect_hecktor_cases_root(str(dest_root)))


DRIVE_HECKTOR_CASES = DRIVE_ROOT / "Trainings" / "Hecktor" / "cases"
HECKTOR_CASES_ROOT = None

# 1) Already mirrored under HECKTOR_ROOT
if all(hecktor_case_ready(HECKTOR_ROOT, c) for c in HECKTOR_CASE_IDS):
    HECKTOR_CASES_ROOT = HECKTOR_ROOT
    print("HECKTOR cases already under", HECKTOR_CASES_ROOT)

# 2) Drive folder: resolve centers by numeric suffix if needed
elif DRIVE_HECKTOR_CASES.is_dir():
    drive_ids = sorted(
        d.name
        for d in DRIVE_HECKTOR_CASES.iterdir()
        if d.is_dir() and not d.name.startswith(".")
    )
    print(f"Drive HECKTOR folders: n={len(drive_ids)}")
    try:
        resolved = [resolve_hecktor_case_id(c, drive_ids) for c in HECKTOR_CASE_IDS]
        if all(hecktor_case_ready(DRIVE_HECKTOR_CASES, c) for c in resolved):
            HECKTOR_CASE_IDS = resolved
            HECKTOR_CASES_ROOT = DRIVE_HECKTOR_CASES
            print("Using Drive HECKTOR cases:", HECKTOR_CASE_IDS)
    except FileNotFoundError as e:
        print("Drive resolve incomplete:", e)

# 3) S3 zip — extract only resolved cases
if HECKTOR_CASES_ROOT is None:
    zip_name = Path(urlparse(HECKTOR_S3_URI).path).name or "hecktor.zip"
    zip_path = download_s3_file(
        HECKTOR_S3_URI, HECKTOR_DOWNLOAD_DIR / zip_name, os.environ["AWS_REGION"]
    )
    available = list_hecktor_case_ids_in_zip(zip_path)
    print(f"HECKTOR IDs in zip: n={len(available)}")
    HECKTOR_CASE_IDS = [resolve_hecktor_case_id(c, available) for c in HECKTOR_CASE_IDS]
    print("Resolved HECKTOR_CASE_IDS:", HECKTOR_CASE_IDS)
    unpack_parent = HECKTOR_DOWNLOAD_DIR / "unzipped_partial"
    found_root = extract_hecktor_cases_from_zip(zip_path, HECKTOR_CASE_IDS, unpack_parent)
    for case_id in HECKTOR_CASE_IDS:
        src = found_root / case_id
        dst = HECKTOR_ROOT / case_id
        if src.is_dir() and not dst.exists():
            shutil.copytree(src, dst)
    HECKTOR_CASES_ROOT = HECKTOR_ROOT
    print("HECKTOR cases ready at", HECKTOR_CASES_ROOT)

for case_id in HECKTOR_CASE_IDS:
    assert hecktor_case_ready(HECKTOR_CASES_ROOT, case_id), f"Missing HECKTOR files for {case_id}"
print("All HECKTOR audit cases OK:", HECKTOR_CASE_IDS)


## 3. Case inventory

Confirm each case resolves to the expected inputs before any mask logic.


In [ ]:
from image_processor.io.file_handler import FileHandler
from image_processor.conventions import get_hecktor_paths

inventory = {"radcure": {}, "hecktor": {}}

for case_id, case_dir in radcure_local.items():
    dicom_folder = FileHandler.get_dicom_path(str(case_dir), case_id)
    paths = FileHandler.get_ct_and_mask_paths(dicom_folder)
    inventory["radcure"][case_id] = {
        "case_dir": str(case_dir),
        "dicom_folder": dicom_folder,
        "ct_path": paths["ct_path"],
        "rtstruct_path": paths["mask_path"],
    }
    print(f"[RADCURE] {case_id}")
    print("  CT:", paths["ct_path"])
    print("  RTSTRUCT:", paths["mask_path"])

for case_id in HECKTOR_CASE_IDS:
    paths = get_hecktor_paths(str(HECKTOR_CASES_ROOT / case_id), case_id)
    inventory["hecktor"][case_id] = paths
    print(f"[HECKTOR] {case_id}")
    print("  CT:", paths["path_ct"])
    print("  Mask:", paths["path_mask"])

print("\nInventory:", {k: list(v) for k, v in inventory.items()})


---

## 4. Pipeline walkthrough (top → bottom)

Same order as `CaseProcessor`, inspect after each stage:

```
RADCURE: DICOM+RTSTRUCT ─┐
                         ├─► CT volume + tumor mask
HECKTOR: NIfTI+mask ─────┘
         ├─► slice crop (tumor ± SLICE_EXPANSION)
         ├─► TotalSegmentator (organs)
         ├─► background / anatomical_region   ← suspect #1
         ├─► organs + leftover → other-tissue ← suspect #1
         ├─► overlay GTVp (/ GTVn)
         ├─► save case_*_0000.nii.gz
         └─► visualisation PDF                ← suspect #2
```


### Step A — Load CT + tumor (no TotalSegmentator yet)

RADCURE: DICOM → NIfTI, then RTSTRUCT **GTVp=1 / GTVn=2** via `load_labeled_tumor_volume`, then **`save_and_align_mask_with_ct`** (same as production — without this, tumor does not sit on the CT).

HECKTOR: load CT + mask NIfTI (already 1=GTVp, 2=GTVn).

Visualisation: **all** selected slices with separate GTVp (red) and GTVn (magenta).


In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from image_processor.io.nifti_handler import NIfTIHandler
from image_processor.core.dicom_handler import DICOMHandler
from image_processor.utils.image_processing import ImageProcessor

nifti_handler = NIfTIHandler()
dicom_handler = DICOMHandler()

loaded = {"radcure": {}, "hecktor": {}}


def tumor_slice_range(mask_vol: np.ndarray, expansion: int) -> list:
    non_zero = ImageProcessor.get_non_zero_slices(mask_vol)
    z = mask_vol.shape[2]
    if not non_zero:
        return list(range(z))
    start = max(int(min(non_zero)) - expansion, 0)
    end = min(int(max(non_zero)) + expansion, z - 1)
    return list(range(start, end + 1))


def summarize_tumor(tumor: np.ndarray) -> str:
    n_p = int(np.sum(tumor == 1))
    n_n = int(np.sum(tumor == 2))
    return f"labels={np.unique(tumor)}, GTVp_voxels={n_p}, GTVn_voxels={n_n}"


# --- RADCURE: convert CT, load separate GTVp/GTVn, ALIGN to CT (required) ---
for case_id, meta in inventory["radcure"].items():
    case_dir = meta["case_dir"]
    ct_nii = nifti_handler.convert_dicom_to_nifti(meta["ct_path"], case_id, case_dir)

    # Separate labels: 1=GTVp, 2=GTVn (GTVn skipped if absent in RTSTRUCT)
    tumor_raw = dicom_handler.load_labeled_tumor_volume(
        meta["ct_path"], meta["rtstruct_path"]
    )
    print(f"[RADCURE] {case_id}: raw RTSTRUCT {summarize_tumor(tumor_raw)} shape={tumor_raw.shape}")

    aligned_path = str(Path(case_dir) / f"{case_id}_tumor_mask_aligned.nii.gz")
    nifti_handler.save_and_align_mask_with_ct(tumor_raw, ct_nii, aligned_path)

    ct = nib.load(ct_nii).get_fdata().astype(np.float32)
    tumor = nib.load(aligned_path).get_fdata().astype(np.int32)
    if tumor.shape != ct.shape:
        raise ValueError(
            f"{case_id}: aligned tumor {tumor.shape} != CT {ct.shape}"
        )

    slices = tumor_slice_range(tumor, SLICE_EXPANSION)
    loaded["radcure"][case_id] = {
        "ct_path": ct_nii,
        "tumor_path": aligned_path,
        "ct": ct,
        "tumor": tumor,
        "slices": slices,
    }
    print(
        f"[RADCURE] {case_id}: aligned CT {ct.shape}, {summarize_tumor(tumor)}, "
        f"slices {slices[0]}..{slices[-1]} (n={len(slices)})"
    )

# --- HECKTOR: already separate GTVp/GTVn on disk ---
for case_id, paths in inventory["hecktor"].items():
    ct = nib.load(paths["path_ct"]).get_fdata().astype(np.float32)
    tumor = nib.load(paths["path_mask"]).get_fdata().astype(np.int32)
    slices = tumor_slice_range(tumor, SLICE_EXPANSION)
    loaded["hecktor"][case_id] = {
        "ct_path": paths["path_ct"],
        "mask_path": paths["path_mask"],
        "ct": ct,
        "tumor": tumor,
        "slices": slices,
    }
    print(
        f"[HECKTOR] {case_id}: CT {ct.shape}, {summarize_tumor(tumor)}, "
        f"slices {slices[0]}..{slices[-1]} (n={len(slices)})"
    )


In [ ]:
def show_ct_tumor_all_slices(
    case_id: str,
    ct: np.ndarray,
    tumor: np.ndarray,
    slices: list,
    title_prefix: str,
    ncols: int = 4,
):
    """All selected slices: CT + separate GTVp (red) / GTVn (magenta)."""
    n = len(slices)
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))

    crop = ct[:, :, slices]
    p1, p99 = np.percentile(crop, (1, 99))
    denom = (p99 - p1) + 1e-8

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax_i, idx in enumerate(slices):
        ax = axes[ax_i]
        ct_s = np.clip((ct[:, :, idx] - p1) / denom, 0, 1)
        m = tumor[:, :, idx]
        ax.imshow(ct_s.T, cmap="gray", origin="lower")
        # Keep GTVp and GTVn separate (do not merge)
        if np.any(m == 1):
            ax.imshow(
                np.ma.masked_where(m != 1, np.ones_like(m)).T,
                cmap="Reds",
                alpha=0.55,
                origin="lower",
                vmin=0,
                vmax=1,
            )
        if np.any(m == 2):
            ax.imshow(
                np.ma.masked_where(m != 2, np.ones_like(m)).T,
                cmap="spring",
                alpha=0.55,
                origin="lower",
                vmin=0,
                vmax=1,
            )
        n_p = int(np.sum(m == 1))
        n_n = int(np.sum(m == 2))
        ax.set_title(f"z={idx} p={n_p} n={n_n}", fontsize=8)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(
        f"{title_prefix} {case_id} — {n} slices | GTVp=red GTVn=magenta",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


for case_id, d in loaded["radcure"].items():
    print(f"RADCURE {case_id}: plotting {len(d['slices'])} slices")
    show_ct_tumor_all_slices(case_id, d["ct"], d["tumor"], d["slices"], "RADCURE")

for case_id, d in loaded["hecktor"].items():
    print(f"HECKTOR {case_id}: plotting {len(d['slices'])} slices")
    show_ct_tumor_all_slices(case_id, d["ct"], d["tumor"], d["slices"], "HECKTOR")


### Step A2 — Anatomy QC (auto-discard non-human / broken FOV)

Heuristic **human-anatomy likelihood** in `[0, 1]` from:
- tumor presence (GTVp preferred; GTVn-only weaker)
- CT intensity dynamic range
- patient fill fraction (`head_mask_from_array` on sample slices)
- body-mask coherence (largest connected component)
- number of selected slices

Cases with `score < ANATOMY_QC_THRESHOLD` are **discarded** from `loaded` for later steps.
All decisions are logged (JSONL); discards also get a CSV summary.


In [ ]:
# Ensure anatomy_qc.py exists in the Colab repo copy (Drive/clone may be stale)
import base64
import os
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
CLONE_DIR = Path("/content/my_tailors_drawer")
REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")

if DRIVE_REPO.is_dir() and (DRIVE_REPO / "image_processor").is_dir():
    REPO_ROOT = DRIVE_REPO
else:
    REPO_ROOT = CLONE_DIR / REPO_SUBPATH

target = REPO_ROOT / "image_processor" / "utils" / "anatomy_qc.py"
target.parent.mkdir(parents=True, exist_ok=True)

_B64 = """IiIiSGV1cmlzdGljIFFDOiBzY29yZSBob3cgbGlrZWx5IGEgQ1QgY3JvcCBsb29rcyBsaWtlIGh1bWFuIGhlYWQvbmVjayBhbmF0b215LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgU2VxdWVuY2UsIFR1cGxlCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBza2ltYWdlIGltcG9ydCBtZWFzdXJlCgpmcm9tIGltYWdlX3Byb2Nlc3Nvci51dGlscy5pbWFnZV9wcm9jZXNzaW5nIGltcG9ydCBJbWFnZVByb2Nlc3NvcgoKCmRlZiBfY2xhbXAwMSh4OiBmbG9hdCkgLT4gZmxvYXQ6CiAgICByZXR1cm4gZmxvYXQobnAuY2xpcCh4LCAwLjAsIDEuMCkpCgoKZGVmIF9iYW5kX3Njb3JlKHZhbHVlOiBmbG9hdCwgbG93OiBmbG9hdCwgaGlnaDogZmxvYXQsIHNvZnQ6IGZsb2F0ID0gMC4xNSkgLT4gZmxvYXQ6CiAgICAiIiIxIGluc2lkZSBbbG93LCBoaWdoXSwgbGluZWFyIGZhbGxvZmYgb3V0c2lkZSBvdmVyIGBgc29mdGBgIHdpZHRoLiIiIgogICAgaWYgbG93IDw9IHZhbHVlIDw9IGhpZ2g6CiAgICAgICAgcmV0dXJuIDEuMAogICAgaWYgdmFsdWUgPCBsb3c6CiAgICAgICAgcmV0dXJuIF9jbGFtcDAxKDEuMCAtIChsb3cgLSB2YWx1ZSkgLyBtYXgoc29mdCwgMWUtNikpCiAgICByZXR1cm4gX2NsYW1wMDEoMS4wIC0gKHZhbHVlIC0gaGlnaCkgLyBtYXgoc29mdCwgMWUtNikpCgoKZGVmIHNjb3JlX2h1bWFuX2FuYXRvbXkoCiAgICBjdDogbnAubmRhcnJheSwKICAgIHR1bW9yOiBucC5uZGFycmF5LAogICAgc2xpY2VzOiBPcHRpb25hbFtTZXF1ZW5jZVtpbnRdXSA9IE5vbmUsCiAgICAqLAogICAgbWluX2d0dnBfdm94ZWxzOiBpbnQgPSAyMCwKICAgIHdlaWdodHM6IE9wdGlvbmFsW0RpY3Rbc3RyLCBmbG9hdF1dID0gTm9uZSwKKSAtPiBEaWN0W3N0ciwgQW55XToKICAgICIiIgogICAgU2NvcmUgbGlrZWxpaG9vZCB0aGF0IGEgdm9sdW1lIGNyb3AgaXMgdXNhYmxlIGh1bWFuIEgmTiBhbmF0b215LgoKICAgIFBhcmFtZXRlcnMKICAgIC0tLS0tLS0tLS0KICAgIGN0LCB0dW1vcgogICAgICAgIDNEIGFycmF5cyAoSCwgVywgWiksIHR1bW9yIGxhYmVscyAxPUdUVnAsIDI9R1RWbi4KICAgIHNsaWNlcwogICAgICAgIFogaW5kaWNlcyB0byBzY29yZSAoZGVmYXVsdDogYWxsKS4KICAgIG1pbl9ndHZwX3ZveGVscwogICAgICAgIFNvZnQgZmxvb3IgZm9yIHByaW1hcnkgdHVtb3IgcHJlc2VuY2UuCiAgICB3ZWlnaHRzCiAgICAgICAgT3B0aW9uYWwgb3ZlcnJpZGUgZm9yIGNvbXBvbmVudCB3ZWlnaHRzIChtdXN0IHN1bSB+MSkuCgogICAgUmV0dXJucwogICAgLS0tLS0tLQogICAgZGljdAogICAgICAgIGBgc2NvcmVgYCBpbiBbMCwgMV0sIGBgY29tcG9uZW50c2BgLCBgYG1ldHJpY3NgYCwgYGBwYXNzYGAgaGVscGVyIGZpZWxkcy4KICAgICIiIgogICAgaWYgc2xpY2VzIGlzIE5vbmU6CiAgICAgICAgc2xpY2VzID0gbGlzdChyYW5nZShjdC5zaGFwZVsyXSkpCiAgICBzbGljZXMgPSBsaXN0KHNsaWNlcykKICAgIGlmIG5vdCBzbGljZXM6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInNjb3JlIjogMC4wLAogICAgICAgICAgICAiY29tcG9uZW50cyI6IHt9LAogICAgICAgICAgICAibWV0cmljcyI6IHsiZXJyb3IiOiAibm9fc2xpY2VzIn0sCiAgICAgICAgICAgICJyZWFzb25zIjogWyJub19zbGljZXMiXSwKICAgICAgICB9CgogICAgY3RfYyA9IGN0WzosIDosIHNsaWNlc10KICAgIHR1X2MgPSB0dW1vcls6LCA6LCBzbGljZXNdCgogICAgZ3R2cCA9IGludChucC5zdW0odHVfYyA9PSAxKSkKICAgIGd0dm4gPSBpbnQobnAuc3VtKHR1X2MgPT0gMikpCiAgICB0dW1vcl9hbnkgPSBpbnQobnAuc3VtKHR1X2MgPiAwKSkKCiAgICAjIC0tLSBjb21wb25lbnQ6IHR1bW9yIHByZXNlbmNlIChwcmVmZXIgR1RWcDsgR1RWbi1vbmx5IGlzIHdlYWtlcikgLS0tCiAgICBpZiBndHZwID49IG1pbl9ndHZwX3ZveGVsczoKICAgICAgICB0dW1vcl9zY29yZSA9IDEuMAogICAgZWxpZiBndHZwID4gMDoKICAgICAgICB0dW1vcl9zY29yZSA9IF9jbGFtcDAxKGd0dnAgLyBmbG9hdChtaW5fZ3R2cF92b3hlbHMpKQogICAgZWxpZiBndHZuID4gMDoKICAgICAgICB0dW1vcl9zY29yZSA9IDAuMzUKICAgIGVsc2U6CiAgICAgICAgdHVtb3Jfc2NvcmUgPSAwLjAKCiAgICAjIC0tLSBjb21wb25lbnQ6IGludGVuc2l0eSBkeW5hbWljIHJhbmdlIG9uIGNyb3AgLS0tCiAgICBwMSwgcDUwLCBwOTkgPSBucC5wZXJjZW50aWxlKGN0X2MuYXN0eXBlKG5wLmZsb2F0NjQpLCAoMSwgNTAsIDk5KSkKICAgIGR5biA9IGZsb2F0KHA5OSAtIHAxKQogICAgIyBGbGF0IC8gZW1wdHkgdm9sdW1lcyBzY29yZSBsb3c7IHR5cGljYWwgQ1QgY3JvcHMgaGF2ZSBkeW4gPj4gMAogICAgaW50ZW5zaXR5X3Njb3JlID0gX2JhbmRfc2NvcmUoZHluLCBsb3c9NTAuMCwgaGlnaD0xZTYsIHNvZnQ9NTAuMCkKICAgIGlmIGR5biA8IDFlLTM6CiAgICAgICAgaW50ZW5zaXR5X3Njb3JlID0gMC4wCgogICAgIyAtLS0gY29tcG9uZW50OiBwYXRpZW50IGZpbGwgdmlhIGV4aXN0aW5nIGhlYWQvYm9keSBoZXVyaXN0aWMgLS0tCiAgICAjIFNhbXBsZSB1cCB0byA1IHNsaWNlcyAoZW5kcyArIG1pZCkgdG8ga2VlcCBRQyBjaGVhcAogICAgc2FtcGxlX2lkeCA9IHNvcnRlZCgKICAgICAgICB7CiAgICAgICAgICAgIHNsaWNlc1swXSwKICAgICAgICAgICAgc2xpY2VzW2xlbihzbGljZXMpIC8vIDRdLAogICAgICAgICAgICBzbGljZXNbbGVuKHNsaWNlcykgLy8gMl0sCiAgICAgICAgICAgIHNsaWNlc1soMyAqIGxlbihzbGljZXMpKSAvLyA0XSwKICAgICAgICAgICAgc2xpY2VzWy0xXSwKICAgICAgICB9CiAgICApCiAgICBmaWxsX2ZyYWNzOiBMaXN0W2Zsb2F0XSA9IFtdCiAgICBsYXJnZXN0X2NjX2ZyYWNzOiBMaXN0W2Zsb2F0XSA9IFtdCiAgICBmb3IgeiBpbiBzYW1wbGVfaWR4OgogICAgICAgICMgaGVhZF9tYXNrX2Zyb21fYXJyYXkgcmV0dXJucyBUcnVlPWJhY2tncm91bmQKICAgICAgICBiZyA9IEltYWdlUHJvY2Vzc29yLmhlYWRfbWFza19mcm9tX2FycmF5KGN0WzosIDosIHpdKQogICAgICAgIHBhdGllbnQgPSB+YmcKICAgICAgICBmaWxsID0gZmxvYXQobnAubWVhbihwYXRpZW50KSkKICAgICAgICBmaWxsX2ZyYWNzLmFwcGVuZChmaWxsKQogICAgICAgICMgY29oZXJlbmNlOiBsYXJnZXN0IENDIC8gcGF0aWVudCBwaXhlbHMgKGlmIGFueSkKICAgICAgICBsYWJlbHMgPSBtZWFzdXJlLmxhYmVsKHBhdGllbnQuYXN0eXBlKG5wLnVpbnQ4KSkKICAgICAgICBpZiBsYWJlbHMubWF4KCkgPT0gMDoKICAgICAgICAgICAgbGFyZ2VzdF9jY19mcmFjcy5hcHBlbmQoMC4wKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNpemVzID0gbnAuYmluY291bnQobGFiZWxzLnJhdmVsKCkpCiAgICAgICAgICAgIHNpemVzWzBdID0gMAogICAgICAgICAgICBsYXJnZXN0X2NjX2ZyYWNzLmFwcGVuZChmbG9hdChzaXplcy5tYXgoKSkgLyBtYXgocGF0aWVudC5zdW0oKSwgMSkpCgogICAgbWVhbl9maWxsID0gZmxvYXQobnAubWVhbihmaWxsX2ZyYWNzKSkgaWYgZmlsbF9mcmFjcyBlbHNlIDAuMAogICAgbWVhbl9jYyA9IGZsb2F0KG5wLm1lYW4obGFyZ2VzdF9jY19mcmFjcykpIGlmIGxhcmdlc3RfY2NfZnJhY3MgZWxzZSAwLjAKICAgICMgSCZOIGF4aWFsOiBwYXRpZW50IG9mdGVuIH4xMOKAkzU1JSBvZiBGT1Y7IG5lYXIgMCBvciB+MSBpcyBzdXNwaWNpb3VzCiAgICBmaWxsX3Njb3JlID0gX2JhbmRfc2NvcmUobWVhbl9maWxsLCBsb3c9MC4wOCwgaGlnaD0wLjYwLCBzb2Z0PTAuMDgpCiAgICBjb2hlcmVuY2Vfc2NvcmUgPSBfYmFuZF9zY29yZShtZWFuX2NjLCBsb3c9MC41NSwgaGlnaD0xLjAsIHNvZnQ9MC4yNSkKCiAgICAjIC0tLSBjb21wb25lbnQ6IGVub3VnaCBheGlhbCBleHRlbnQgLS0tCiAgICBuX3NsaWNlcyA9IGxlbihzbGljZXMpCiAgICBleHRlbnRfc2NvcmUgPSBfYmFuZF9zY29yZShmbG9hdChuX3NsaWNlcyksIGxvdz04LjAsIGhpZ2g9MjAwLjAsIHNvZnQ9Ni4wKQoKICAgIGNvbXBvbmVudHMgPSB7CiAgICAgICAgInR1bW9yIjogdHVtb3Jfc2NvcmUsCiAgICAgICAgImludGVuc2l0eSI6IGludGVuc2l0eV9zY29yZSwKICAgICAgICAicGF0aWVudF9maWxsIjogZmlsbF9zY29yZSwKICAgICAgICAiYm9keV9jb2hlcmVuY2UiOiBjb2hlcmVuY2Vfc2NvcmUsCiAgICAgICAgInNsaWNlX2V4dGVudCI6IGV4dGVudF9zY29yZSwKICAgIH0KICAgIHcgPSB3ZWlnaHRzIG9yIHsKICAgICAgICAidHVtb3IiOiAwLjMwLAogICAgICAgICJpbnRlbnNpdHkiOiAwLjE1LAogICAgICAgICJwYXRpZW50X2ZpbGwiOiAwLjI1LAogICAgICAgICJib2R5X2NvaGVyZW5jZSI6IDAuMjAsCiAgICAgICAgInNsaWNlX2V4dGVudCI6IDAuMTAsCiAgICB9CiAgICB3c3VtID0gc3VtKHcuZ2V0KGssIDAuMCkgZm9yIGsgaW4gY29tcG9uZW50cykgb3IgMS4wCiAgICBzY29yZSA9IGZsb2F0KHN1bShjb21wb25lbnRzW2tdICogdy5nZXQoaywgMC4wKSBmb3IgayBpbiBjb21wb25lbnRzKSAvIHdzdW0pCgogICAgcmVhc29uczogTGlzdFtzdHJdID0gW10KICAgIGlmIHR1bW9yX3Njb3JlIDwgMC41OgogICAgICAgIHJlYXNvbnMuYXBwZW5kKCJ3ZWFrX29yX21pc3NpbmdfdHVtb3IiKQogICAgaWYgZmlsbF9zY29yZSA8IDAuNToKICAgICAgICByZWFzb25zLmFwcGVuZChmImFibm9ybWFsX3BhdGllbnRfZmlsbD17bWVhbl9maWxsOi4zZn0iKQogICAgaWYgY29oZXJlbmNlX3Njb3JlIDwgMC41OgogICAgICAgIHJlYXNvbnMuYXBwZW5kKGYiZnJhZ21lbnRlZF9ib2R5X21hc2tfY2M9e21lYW5fY2M6LjNmfSIpCiAgICBpZiBpbnRlbnNpdHlfc2NvcmUgPCAwLjU6CiAgICAgICAgcmVhc29ucy5hcHBlbmQoZiJmbGF0X2ludGVuc2l0eV9keW49e2R5bjouM2Z9IikKICAgIGlmIGV4dGVudF9zY29yZSA8IDAuNToKICAgICAgICByZWFzb25zLmFwcGVuZChmImZld19zbGljZXM9e25fc2xpY2VzfSIpCgogICAgcmV0dXJuIHsKICAgICAgICAic2NvcmUiOiBzY29yZSwKICAgICAgICAiY29tcG9uZW50cyI6IGNvbXBvbmVudHMsCiAgICAgICAgIm1ldHJpY3MiOiB7CiAgICAgICAgICAgICJndHZwX3ZveGVscyI6IGd0dnAsCiAgICAgICAgICAgICJndHZuX3ZveGVscyI6IGd0dm4sCiAgICAgICAgICAgICJ0dW1vcl92b3hlbHMiOiB0dW1vcl9hbnksCiAgICAgICAgICAgICJuX3NsaWNlcyI6IG5fc2xpY2VzLAogICAgICAgICAgICAiY3RfcDEiOiBmbG9hdChwMSksCiAgICAgICAgICAgICJjdF9wNTAiOiBmbG9hdChwNTApLAogICAgICAgICAgICAiY3RfcDk5IjogZmxvYXQocDk5KSwKICAgICAgICAgICAgImN0X2R5bmFtaWNfcmFuZ2UiOiBkeW4sCiAgICAgICAgICAgICJtZWFuX3BhdGllbnRfZmlsbCI6IG1lYW5fZmlsbCwKICAgICAgICAgICAgIm1lYW5fbGFyZ2VzdF9jY19mcmFjIjogbWVhbl9jYywKICAgICAgICAgICAgInNhbXBsZV96Ijogc2FtcGxlX2lkeCwKICAgICAgICB9LAogICAgICAgICJyZWFzb25zIjogcmVhc29ucywKICAgIH0KCgpkZWYgYXBwbHlfYW5hdG9teV90aHJlc2hvbGQoCiAgICBzY29yZV9yZXN1bHQ6IERpY3Rbc3RyLCBBbnldLAogICAgdGhyZXNob2xkOiBmbG9hdCA9IDAuNTUsCikgLT4gVHVwbGVbYm9vbCwgRGljdFtzdHIsIEFueV1dOgogICAgIiIiUmV0dXJuIChrZWVwLCByZWNvcmQpLiBrZWVwPVRydWUgaWYgc2NvcmUgPj0gdGhyZXNob2xkLiIiIgogICAga2VlcCA9IGZsb2F0KHNjb3JlX3Jlc3VsdC5nZXQoInNjb3JlIiwgMC4wKSkgPj0gZmxvYXQodGhyZXNob2xkKQogICAgcmVjb3JkID0gewogICAgICAgICoqc2NvcmVfcmVzdWx0LAogICAgICAgICJ0aHJlc2hvbGQiOiBmbG9hdCh0aHJlc2hvbGQpLAogICAgICAgICJrZWVwIjoga2VlcCwKICAgICAgICAiZGVjaXNpb24iOiAia2VlcCIgaWYga2VlcCBlbHNlICJkaXNjYXJkIiwKICAgIH0KICAgIHJldHVybiBrZWVwLCByZWNvcmQKCgpkZWYgYXBwZW5kX3FjX2xvZygKICAgIGxvZ19wYXRoOiBzdHIsCiAgICAqLAogICAgY2FzZV9pZDogc3RyLAogICAgY29udmVudGlvbjogc3RyLAogICAgcmVjb3JkOiBEaWN0W3N0ciwgQW55XSwKICAgIGV4dHJhOiBPcHRpb25hbFtEaWN0W3N0ciwgQW55XV0gPSBOb25lLAopIC0+IE5vbmU6CiAgICAiIiJBcHBlbmQgb25lIEpTT04gbGluZSBmb3IgYSBjYXNlIFFDIGRlY2lzaW9uIChrZXB0IG9yIGRpc2NhcmRlZCkuIiIiCiAgICBvcy5tYWtlZGlycyhvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKGxvZ19wYXRoKSkgb3IgIi4iLCBleGlzdF9vaz1UcnVlKQogICAgcm93ID0gewogICAgICAgICJ0c191dGMiOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKSwKICAgICAgICAiY2FzZV9pZCI6IGNhc2VfaWQsCiAgICAgICAgImNvbnZlbnRpb24iOiBjb252ZW50aW9uLAogICAgICAgICoqcmVjb3JkLAogICAgfQogICAgaWYgZXh0cmE6CiAgICAgICAgcm93WyJleHRyYSJdID0gZXh0cmEKICAgIHdpdGggb3Blbihsb2dfcGF0aCwgImEiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGYud3JpdGUoanNvbi5kdW1wcyhyb3csIGRlZmF1bHQ9ZmxvYXQpICsgIlxuIikKCgpkZWYgd3JpdGVfZGlzY2FyZF9zdW1tYXJ5X2NzdihkaXNjYXJkX3JlY29yZHM6IExpc3RbRGljdFtzdHIsIEFueV1dLCBjc3ZfcGF0aDogc3RyKSAtPiBOb25lOgogICAgIiIiV3JpdGUgYSBzbWFsbCBDU1Ygb2YgZGlzY2FyZGVkIGNhc2VzIGZvciBxdWljayByZXZpZXcuIiIiCiAgICBpbXBvcnQgY3N2CgogICAgb3MubWFrZWRpcnMob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChjc3ZfcGF0aCkpIG9yICIuIiwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGZpZWxkcyA9IFsKICAgICAgICAiY2FzZV9pZCIsCiAgICAgICAgImNvbnZlbnRpb24iLAogICAgICAgICJzY29yZSIsCiAgICAgICAgInRocmVzaG9sZCIsCiAgICAgICAgInJlYXNvbnMiLAogICAgICAgICJndHZwX3ZveGVscyIsCiAgICAgICAgImd0dm5fdm94ZWxzIiwKICAgICAgICAibWVhbl9wYXRpZW50X2ZpbGwiLAogICAgICAgICJjdF9keW5hbWljX3JhbmdlIiwKICAgIF0KICAgIHdpdGggb3Blbihjc3ZfcGF0aCwgInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHcgPSBjc3YuRGljdFdyaXRlcihmLCBmaWVsZG5hbWVzPWZpZWxkcykKICAgICAgICB3LndyaXRlaGVhZGVyKCkKICAgICAgICBmb3IgciBpbiBkaXNjYXJkX3JlY29yZHM6CiAgICAgICAgICAgIG0gPSByLmdldCgibWV0cmljcyIpIG9yIHt9CiAgICAgICAgICAgIHcud3JpdGVyb3coCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImNhc2VfaWQiOiByLmdldCgiY2FzZV9pZCIsICIiKSwKICAgICAgICAgICAgICAgICAgICAiY29udmVudGlvbiI6IHIuZ2V0KCJjb252ZW50aW9uIiwgIiIpLAogICAgICAgICAgICAgICAgICAgICJzY29yZSI6IHIuZ2V0KCJzY29yZSIsICIiKSwKICAgICAgICAgICAgICAgICAgICAidGhyZXNob2xkIjogci5nZXQoInRocmVzaG9sZCIsICIiKSwKICAgICAgICAgICAgICAgICAgICAicmVhc29ucyI6ICI7Ii5qb2luKHIuZ2V0KCJyZWFzb25zIikgb3IgW10pLAogICAgICAgICAgICAgICAgICAgICJndHZwX3ZveGVscyI6IG0uZ2V0KCJndHZwX3ZveGVscyIsICIiKSwKICAgICAgICAgICAgICAgICAgICAiZ3R2bl92b3hlbHMiOiBtLmdldCgiZ3R2bl92b3hlbHMiLCAiIiksCiAgICAgICAgICAgICAgICAgICAgIm1lYW5fcGF0aWVudF9maWxsIjogbS5nZXQoIm1lYW5fcGF0aWVudF9maWxsIiwgIiIpLAogICAgICAgICAgICAgICAgICAgICJjdF9keW5hbWljX3JhbmdlIjogbS5nZXQoImN0X2R5bmFtaWNfcmFuZ2UiLCAiIiksCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkK"""
target.write_bytes(base64.b64decode(_B64))
print("Wrote", target, "bytes=", target.stat().st_size)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Drop cached imports
for mod in list(sys.modules):
    if mod == "image_processor.utils.anatomy_qc" or mod.startswith("image_processor.utils.anatomy_qc"):
        del sys.modules[mod]

from image_processor.utils.anatomy_qc import score_human_anatomy
print("Import OK:", score_human_anatomy)


In [ ]:
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

from image_processor.utils.anatomy_qc import (
    score_human_anatomy,
    apply_anatomy_threshold,
    append_qc_log,
    write_discard_summary_csv,
)

# Tune after inspecting scores on the audit set
ANATOMY_QC_THRESHOLD = 0.70

QC_DIR = WORK_DIR / "logs" / "anatomy_qc"
QC_DIR.mkdir(parents=True, exist_ok=True)
QC_JSONL = QC_DIR / "anatomy_qc_decisions.jsonl"
QC_DISCARD_CSV = QC_DIR / "anatomy_qc_discarded.csv"

# Fresh run: optional wipe of previous JSONL for this session
# QC_JSONL.write_text("")

qc_rows = []
discard_rows = []
kept = {"radcure": {}, "hecktor": {}}

for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        result = score_human_anatomy(d["ct"], d["tumor"], d["slices"])
        keep, record = apply_anatomy_threshold(result, threshold=ANATOMY_QC_THRESHOLD)
        record["case_id"] = case_id
        record["convention"] = convention
        append_qc_log(str(QC_JSONL), case_id=case_id, convention=convention, record=record)
        qc_rows.append(record)

        comps = record["components"]
        print(
            f"[{convention}] {case_id}: score={record['score']:.3f} "
            f"→ {record['decision']} hard_fail={record.get('hard_fail')} | "
            f"tumor={comps['tumor']:.2f} fill={comps['patient_fill']:.2f} "
            f"in_pat={comps.get('tumor_inside_patient', 0):.2f} "
            f"coh={comps['body_coherence']:.2f}"
        )
        if record["reasons"]:
            print(f"    reasons: {record['reasons']}")

        if keep:
            kept[convention][case_id] = d
        else:
            discard_rows.append(record)

# Restrict downstream work to kept cases
loaded = kept
n_keep = sum(len(v) for v in loaded.values())
n_drop = len(discard_rows)
print(f"\nKept {n_keep} cases, discarded {n_drop}")
print("Log:", QC_JSONL)

if discard_rows:
    write_discard_summary_csv(discard_rows, str(QC_DISCARD_CSV))
    print("Discard CSV:", QC_DISCARD_CSV)
else:
    print("No discards at threshold", ANATOMY_QC_THRESHOLD)

# Quick table
summary = []
# Re-read decisions from this run
for r in qc_rows:
    summary.append({
        "convention": r["convention"],
        "case_id": r["case_id"],
        "score": round(r["score"], 3),
        "decision": r["decision"],
        "gtvp": r["metrics"].get("gtvp_voxels"),
        "fill": round(r["metrics"].get("mean_patient_fill", 0), 3),
        "hard_fail": r.get("hard_fail"),
        "tumor_in_pat": round(r["metrics"].get("tumor_inside_patient_frac", 0), 3),
        "reasons": ";".join(r.get("reasons") or []),
    })
if pd is not None:
    display(pd.DataFrame(summary).sort_values(["decision", "score"]))
else:
    for row in sorted(summary, key=lambda r: (r["decision"], r["score"])):
        print(row)


### Step A3 — Extra audit cases (3 RADCURE + 2 HECKTOR)

New cases (not in the first batch). Run after Step A / A2 setup is working.

| Cohort | Case IDs |
|--------|----------|
| RADCURE | `RADCURE-0005`, `RADCURE-0088`, `RADCURE-0250` |
| HECKTOR | `CHUM-013`, `CHUS-016` *(from available test1.zip centers: CHUM/CHUS)* |

Flow: define IDs → download → load+align (Step A) → optional QC (A2).


In [ ]:
# --- Extra case IDs (batch 2) ---
EXTRA_RADCURE_CASE_IDS = ["RADCURE-0005", "RADCURE-0088", "RADCURE-0250"]
EXTRA_HECKTOR_CASE_IDS = ["CHUM-013", "CHUS-016"]  # provisional; may resolve by numeric suffix

print("Extra RADCURE:", EXTRA_RADCURE_CASE_IDS)
print("Extra HECKTOR:", EXTRA_HECKTOR_CASE_IDS)


In [ ]:
# --- Download extra RADCURE ---
from image_processor.io.aws_handler import AWSHandler
from image_processor.io.file_handler import FileHandler

aws = AWSHandler(
    bucket_name=AWS_BUCKET_NAME,
    aws_folder=AWS_FOLDER,
    region_name=os.environ["AWS_REGION"],
)

if "radcure_local" not in dir():
    radcure_local = {}

for case_id in EXTRA_RADCURE_CASE_IDS:
    case_dir = RADCURE_ROOT / case_id
    zip_path = RADCURE_ROOT / f"{case_id}.zip"
    if not case_dir.is_dir():
        if not zip_path.is_file():
            print(f"Downloading {case_id} ...")
            aws.download_case(case_id, str(RADCURE_ROOT))
        print(f"Unzipping {case_id} ...")
        FileHandler.unzip_file(str(zip_path), str(case_dir))
    else:
        print(f"Already present: {case_dir}")
    radcure_local[case_id] = case_dir

print("RADCURE local now:", sorted(radcure_local))


In [ ]:
# --- Download extra HECKTOR ---
import os
import shutil
import zipfile
from pathlib import Path
from urllib.parse import urlparse

from image_processor.conventions import get_hecktor_paths


def resolve_hecktor_case_id(preferred_id: str, available_ids: list) -> str:
    if preferred_id in available_ids:
        return preferred_id
    num = preferred_id.split("-")[-1]
    matches = [a for a in available_ids if a.split("-")[-1] == num]
    if not matches:
        raise FileNotFoundError(
            f"No HECKTOR case for {preferred_id} (suffix {num}). sample={available_ids[:15]}"
        )
    chosen = sorted(matches)[0]
    print(f"  {preferred_id} not found -> using {chosen}")
    return chosen


def hecktor_case_ready(cases_root: Path, case_id: str) -> bool:
    paths = get_hecktor_paths(str(cases_root / case_id), case_id)
    return os.path.isfile(paths["path_ct"]) and os.path.isfile(paths["path_mask"])


def list_hecktor_case_ids_in_zip(zip_path: Path) -> list:
    ids = set()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for n in zf.namelist():
            for part in Path(n).parts:
                if part.endswith(".nii.gz") or part in ("", ".", "__MACOSX"):
                    continue
                if "-" in part:
                    tail = part.split("-")[-1]
                    if tail.isdigit() and 1 <= len(tail) <= 4:
                        ids.add(part)
    return sorted(ids)


def download_s3_file(s3_uri: str, local_path: Path, region: str = "eu-west-1") -> Path:
    import boto3
    parsed = urlparse(s3_uri)
    bucket, key = parsed.netloc, parsed.path.lstrip("/")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_file():
        print("Zip already on disk:", local_path)
        return local_path
    print(f"Downloading {s3_uri} -> {local_path}")
    boto3.client("s3", region_name=region).download_file(bucket, key, str(local_path))
    return local_path


def extract_hecktor_cases_from_zip(zip_path: Path, case_ids: list, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        for case_id in case_ids:
            members = [n for n in names if f"/{case_id}/" in f"/{n}" or n.startswith(f"{case_id}/")]
            if not members:
                raise FileNotFoundError(f"{case_id} not found in {zip_path.name}")
            print(f"Extracting {case_id}: {len(members)} files")
            for m in members:
                zf.extract(m, dest_root)
    from pipelines.hecktor.test_pipeline import detect_hecktor_cases_root
    return Path(detect_hecktor_cases_root(str(dest_root)))


if "HECKTOR_CASES_ROOT" not in dir() or HECKTOR_CASES_ROOT is None:
    HECKTOR_CASES_ROOT = HECKTOR_ROOT

DRIVE_HECKTOR_CASES = DRIVE_ROOT / "Trainings" / "Hecktor" / "cases"
final_ids = []
need_from_zip = []

for cid in EXTRA_HECKTOR_CASE_IDS:
    if hecktor_case_ready(Path(HECKTOR_CASES_ROOT), cid):
        final_ids.append(cid)
        print("Already ready:", cid)
        continue
    if DRIVE_HECKTOR_CASES.is_dir():
        drive_ids = sorted(
            d.name for d in DRIVE_HECKTOR_CASES.iterdir()
            if d.is_dir() and not d.name.startswith(".")
        )
        try:
            rid = resolve_hecktor_case_id(cid, drive_ids)
            if hecktor_case_ready(DRIVE_HECKTOR_CASES, rid):
                final_ids.append(rid)
                HECKTOR_CASES_ROOT = DRIVE_HECKTOR_CASES
                print("Drive:", rid)
                continue
        except FileNotFoundError:
            pass
    need_from_zip.append(cid)

if need_from_zip:
    zip_name = Path(urlparse(HECKTOR_S3_URI).path).name or "hecktor.zip"
    zip_path = download_s3_file(
        HECKTOR_S3_URI, HECKTOR_DOWNLOAD_DIR / zip_name, os.environ["AWS_REGION"]
    )
    available = list_hecktor_case_ids_in_zip(zip_path)
    print(f"HECKTOR IDs in zip: n={len(available)}")
    to_extract = [resolve_hecktor_case_id(cid, available) for cid in need_from_zip]
    unpack_parent = HECKTOR_DOWNLOAD_DIR / "unzipped_partial"
    found_root = extract_hecktor_cases_from_zip(zip_path, to_extract, unpack_parent)
    for case_id in to_extract:
        src = found_root / case_id
        dst = HECKTOR_ROOT / case_id
        if src.is_dir() and not dst.exists():
            shutil.copytree(src, dst)
        final_ids.append(case_id)
    HECKTOR_CASES_ROOT = HECKTOR_ROOT

EXTRA_HECKTOR_CASE_IDS = final_ids
print("Extra HECKTOR resolved:", EXTRA_HECKTOR_CASE_IDS)
for case_id in EXTRA_HECKTOR_CASE_IDS:
    assert hecktor_case_ready(Path(HECKTOR_CASES_ROOT), case_id), case_id
print("All extra HECKTOR OK")


In [ ]:
# --- Step A load for EXTRA cases only (align RADCURE; keep GTVp/GTVn separate) ---
import numpy as np
import nibabel as nib
from pathlib import Path

from image_processor.io.nifti_handler import NIfTIHandler
from image_processor.core.dicom_handler import DICOMHandler
from image_processor.utils.image_processing import ImageProcessor
from image_processor.io.file_handler import FileHandler
from image_processor.conventions import get_hecktor_paths

nifti_handler = NIfTIHandler()
dicom_handler = DICOMHandler()

if "loaded" not in dir():
    loaded = {"radcure": {}, "hecktor": {}}


def tumor_slice_range(mask_vol, expansion):
    non_zero = ImageProcessor.get_non_zero_slices(mask_vol)
    z = mask_vol.shape[2]
    if not non_zero:
        return list(range(z))
    start = max(int(min(non_zero)) - expansion, 0)
    end = min(int(max(non_zero)) + expansion, z - 1)
    return list(range(start, end + 1))


def summarize_tumor(tumor):
    return (
        f"labels={np.unique(tumor)}, "
        f"GTVp={int(np.sum(tumor == 1))}, GTVn={int(np.sum(tumor == 2))}"
    )


# Inventory paths for extras
for case_id in EXTRA_RADCURE_CASE_IDS:
    case_dir = radcure_local[case_id]
    dicom_folder = FileHandler.get_dicom_path(str(case_dir), case_id)
    paths = FileHandler.get_ct_and_mask_paths(dicom_folder)
    ct_nii = nifti_handler.convert_dicom_to_nifti(paths["ct_path"], case_id, str(case_dir))
    tumor_raw = dicom_handler.load_labeled_tumor_volume(paths["ct_path"], paths["mask_path"])
    aligned_path = str(Path(case_dir) / f"{case_id}_tumor_mask_aligned.nii.gz")
    nifti_handler.save_and_align_mask_with_ct(tumor_raw, ct_nii, aligned_path)
    ct = nib.load(ct_nii).get_fdata().astype(np.float32)
    tumor = nib.load(aligned_path).get_fdata().astype(np.int32)
    slices = tumor_slice_range(tumor, SLICE_EXPANSION)
    loaded["radcure"][case_id] = {
        "ct_path": ct_nii,
        "tumor_path": aligned_path,
        "ct": ct,
        "tumor": tumor,
        "slices": slices,
    }
    print(f"[RADCURE extra] {case_id}: {ct.shape}, {summarize_tumor(tumor)}, n_slices={len(slices)}")

for case_id in EXTRA_HECKTOR_CASE_IDS:
    paths = get_hecktor_paths(str(HECKTOR_CASES_ROOT / case_id), case_id)
    ct = nib.load(paths["path_ct"]).get_fdata().astype(np.float32)
    tumor = nib.load(paths["path_mask"]).get_fdata().astype(np.int32)
    slices = tumor_slice_range(tumor, SLICE_EXPANSION)
    loaded["hecktor"][case_id] = {
        "ct_path": paths["path_ct"],
        "mask_path": paths["path_mask"],
        "ct": ct,
        "tumor": tumor,
        "slices": slices,
    }
    print(f"[HECKTOR extra] {case_id}: {ct.shape}, {summarize_tumor(tumor)}, n_slices={len(slices)}")

print("loaded counts:", {k: len(v) for k, v in loaded.items()})


In [ ]:
# --- Quick viz for EXTRA cases only (all selected slices) ---
import matplotlib.pyplot as plt

def show_ct_tumor_all_slices(case_id, ct, tumor, slices, title_prefix, ncols=4):
    n = len(slices)
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    crop = ct[:, :, slices]
    p1, p99 = np.percentile(crop, (1, 99))
    denom = (p99 - p1) + 1e-8
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax_i, idx in enumerate(slices):
        ax = axes[ax_i]
        ct_s = np.clip((ct[:, :, idx] - p1) / denom, 0, 1)
        m = tumor[:, :, idx]
        ax.imshow(ct_s.T, cmap="gray", origin="lower")
        if np.any(m == 1):
            ax.imshow(np.ma.masked_where(m != 1, np.ones_like(m)).T, cmap="Reds", alpha=0.55, origin="lower", vmin=0, vmax=1)
        if np.any(m == 2):
            ax.imshow(np.ma.masked_where(m != 2, np.ones_like(m)).T, cmap="spring", alpha=0.55, origin="lower", vmin=0, vmax=1)
        ax.set_title(f"z={idx} p={int((m==1).sum())} n={int((m==2).sum())}", fontsize=8)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(f"{title_prefix} {case_id} — EXTRA | GTVp=red GTVn=magenta", fontsize=12)
    plt.tight_layout()
    plt.show()

for case_id in EXTRA_RADCURE_CASE_IDS:
    d = loaded["radcure"][case_id]
    show_ct_tumor_all_slices(case_id, d["ct"], d["tumor"], d["slices"], "RADCURE")

for case_id in EXTRA_HECKTOR_CASE_IDS:
    d = loaded["hecktor"][case_id]
    show_ct_tumor_all_slices(case_id, d["ct"], d["tumor"], d["slices"], "HECKTOR")


In [ ]:
# --- QC on EXTRA cases only (uses improved anatomy_qc) ---
from image_processor.utils.anatomy_qc import (
    score_human_anatomy,
    apply_anatomy_threshold,
    append_qc_log,
    write_discard_summary_csv,
)

ANATOMY_QC_THRESHOLD = 0.70
QC_DIR = WORK_DIR / "logs" / "anatomy_qc"
QC_DIR.mkdir(parents=True, exist_ok=True)
QC_JSONL = QC_DIR / "anatomy_qc_decisions.jsonl"
QC_DISCARD_CSV = QC_DIR / "anatomy_qc_discarded_extra.csv"

extra_ids = {
    "radcure": list(EXTRA_RADCURE_CASE_IDS),
    "hecktor": list(EXTRA_HECKTOR_CASE_IDS),
}

qc_rows = []
discard_rows = []
for convention, ids in extra_ids.items():
    for case_id in ids:
        d = loaded[convention][case_id]
        result = score_human_anatomy(d["ct"], d["tumor"], d["slices"])
        keep, record = apply_anatomy_threshold(result, threshold=ANATOMY_QC_THRESHOLD)
        record["case_id"] = case_id
        record["convention"] = convention
        append_qc_log(str(QC_JSONL), case_id=case_id, convention=convention, record=record)
        qc_rows.append(record)
        comps = record["components"]
        print(
            f"[{convention}] {case_id}: score={record['score']:.3f} → {record['decision']} "
            f"hard_fail={record.get('hard_fail')} | fill={comps['patient_fill']:.2f} "
            f"in_pat={comps.get('tumor_inside_patient', 0):.2f}"
        )
        if record["reasons"]:
            print("   ", record["reasons"])
        if not keep:
            discard_rows.append(record)
            # optional: drop from loaded
            # del loaded[convention][case_id]

if discard_rows:
    write_discard_summary_csv(discard_rows, str(QC_DISCARD_CSV))
    print("Discard CSV:", QC_DISCARD_CSV)

try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame([{
        "convention": r["convention"],
        "case_id": r["case_id"],
        "score": round(r["score"], 3),
        "decision": r["decision"],
        "hard_fail": r.get("hard_fail"),
        "fill": round(r["metrics"].get("mean_patient_fill", 0), 3),
        "gtvp": r["metrics"].get("gtvp_voxels"),
        "reasons": ";".join(r.get("reasons") or []),
    } for r in qc_rows]).sort_values(["decision", "score"]))
except Exception:
    for r in qc_rows:
        print(r["case_id"], r["score"], r["decision"], r.get("reasons"))


### Step B — Background / anatomical_region

Goals:
1. Drop QC-discarded cases
2. Head **and shoulders**, centered body, always leave background
3. **Z continuity** — fill missing slices from neighbors
4. **Sagittal L/R symmetry** — flip along display left-right (`flip_axis=0` with `imshow(.T)`).  
   Not A/P (that was the table/top artifact from `fliplr`).
5. Symmetry fill only where **body vs air contrast** supports it, inside the patient bbox.

Row1 = production · Row2 = raw intensity · Row3 = improved (sym L/R + Z).


In [ ]:
# --- Step B0: keep only QC-passed cases in `loaded` ---
from image_processor.utils.anatomy_qc import score_human_anatomy, apply_anatomy_threshold

ANATOMY_QC_THRESHOLD = 0.70

kept = {"radcure": {}, "hecktor": {}}
dropped = []
for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        result = score_human_anatomy(d["ct"], d["tumor"], d["slices"])
        keep, record = apply_anatomy_threshold(result, threshold=ANATOMY_QC_THRESHOLD)
        if keep:
            kept[convention][case_id] = d
        else:
            dropped.append((convention, case_id, round(record["score"], 3), record.get("reasons")))

loaded = kept
print(f"Kept for Step B: {sum(len(v) for v in loaded.values())}")
print("Dropped (QC):")
for row in dropped:
    print(" ", row)


In [ ]:
# --- Step B1: production vs raw intensity vs improved (sym + Z) ---
import numpy as np
import inspect
from image_processor.utils.image_processing import ImageProcessor

# Sanity: confirm Colab loaded the NEW continuity code
src = inspect.getsource(ImageProcessor.enforce_anatomical_continuity)
assert "z_radius" in src, "Stale continuity code — pull + reinstall"
sym_src = inspect.getsource(ImageProcessor._enforce_sagittal_symmetry)
assert "flip_axis" in sym_src and "flipud" in sym_src, (
    "Stale symmetry code — pull + reinstall"
)
print("OK: new continuity + sagittal flip_axis symmetry loaded")


def compute_production_bg(ct, slices):
    return [
        np.where(ImageProcessor.head_mask_from_array(ct[:, :, z]), 0, 1).astype(np.int32)
        for z in slices
    ]


def count_empty(masks, thr=1e-6):
    fills = [float(np.mean(m == 1)) for m in masks]
    return int(sum(f < thr for f in fills)), float(np.mean(fills)), fills


assert sum(len(v) for v in loaded.values()) > 0, "No kept cases — run B0 first"

for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        bg_prod = compute_production_bg(d["ct"], d["slices"])
        bg_raw = [
            np.where(
                ImageProcessor.body_mask_from_intensity(
                    d["ct"][:, :, z], enforce_symmetry=False
                ),
                0,
                1,
            ).astype(np.int32)
            for z in d["slices"]
        ]
        bg_imp = ImageProcessor.anatomical_region_masks_from_slices(
            d["ct"], d["slices"],
            enforce_symmetry=True, enforce_continuity=True,
            sagittal_flip_axis=0,  # L/R after imshow(.T)
        )

        pe, pm, pf = count_empty(bg_prod)
        re, rm, rf = count_empty(bg_raw)
        ie, imean, iff = count_empty(bg_imp)

        d["background_production"] = bg_prod
        d["background_intensity_raw"] = bg_raw
        d["background_intensity"] = bg_imp
        d["background_production_fill_mean"] = pm
        d["background_production_fill_per_slice"] = pf
        d["background_intensity_fill_mean"] = imean
        d["background_intensity_fill_per_slice"] = iff

        print(
            f"[{convention}] {case_id}: empty slices  "
            f"prod={pe}  raw={re}  improved={ie}  |  "
            f"mean_fill prod={pm:.3f} raw={rm:.3f} improved={imean:.3f}"
        )
        if re > 0 and ie >= re:
            print("  WARNING: continuity did not reduce empty slices — check pull/install")


In [ ]:
# --- Step B2: 3 rows (prod | raw | improved), CONSECUTIVE slices (not linspace) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def _overlay_rgba(ax, ct_s, anat, tumor, title: str):
    ax.imshow(ct_s.T, cmap="gray", origin="lower")
    h, w = anat.shape
    rgba = np.zeros((h, w, 4), dtype=np.float32)
    rgba[anat == 1] = (0.0, 0.85, 0.85, 0.40)
    rgba[tumor == 1] = (1.0, 0.0, 0.0, 0.65)
    rgba[tumor == 2] = (1.0, 0.0, 1.0, 0.65)
    ax.imshow(np.transpose(rgba, (1, 0, 2)), origin="lower")
    ax.set_title(f"{title} f={float(np.mean(anat == 1)):.2f}", fontsize=8)
    ax.axis("off")


def show_bg_three_rows(case_id, ct, tumor, slices, bg_prod, bg_raw, bg_imp, title_prefix, n_show=10):
    """Show a consecutive window of slices so Z-continuity is visible."""
    n_all = len(slices)
    n_show = min(n_show, n_all)
    start = max(0, (n_all - n_show) // 2)
    sel = list(range(start, start + n_show))
    idxs = [slices[i] for i in sel]
    rows_masks = [
        [bg_prod[i] for i in sel],
        [bg_raw[i] for i in sel],
        [bg_imp[i] for i in sel],
    ]
    row_names = ["prod (broken)", "raw inten", "improved sym+Z"]

    crop = ct[:, :, idxs]
    p1, p99 = np.percentile(crop, (1, 99))
    denom = (p99 - p1) + 1e-8

    fig, axes = plt.subplots(3, n_show, figsize=(2.4 * n_show, 7.2))
    if n_show == 1:
        axes = np.array(axes).reshape(3, 1)

    for r, (name, masks) in enumerate(zip(row_names, rows_masks)):
        for j, (z, m) in enumerate(zip(idxs, masks)):
            ct_s = np.clip((ct[:, :, z] - p1) / denom, 0, 1)
            _overlay_rgba(axes[r, j], ct_s, m, tumor[:, :, z], f"z={z}")
        axes[r, 0].set_ylabel(name, fontsize=9)

    legend = [
        mpatches.Patch(facecolor=(0.0, 0.85, 0.85), label="anatomical"),
        mpatches.Patch(facecolor=(1.0, 0.0, 0.0), label="GTVp"),
        mpatches.Patch(facecolor=(1.0, 0.0, 1.0), label="GTVn"),
    ]
    fig.legend(handles=legend, loc="upper right", fontsize=8)
    fig.suptitle(
        f"{title_prefix} {case_id} — consecutive z={idxs[0]}..{idxs[-1]} | "
        f"compare row3 vs row1/2",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


for convention, label in (("radcure", "RADCURE"), ("hecktor", "HECKTOR")):
    for case_id, d in loaded[convention].items():
        print(f"Plotting [{convention}] {case_id}")
        show_bg_three_rows(
            case_id, d["ct"], d["tumor"], d["slices"],
            d["background_production"],
            d["background_intensity_raw"],
            d["background_intensity"],
            label,
            n_show=10,
        )


In [ ]:
# --- Step B3: summary ---
rows = []
for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        inten_fills = d.get("background_intensity_fill_per_slice", [])
        prod_fills = d.get("background_production_fill_per_slice", [])
        rows.append({
            "convention": convention,
            "case_id": case_id,
            "prod_fill": round(d.get("background_production_fill_mean", float("nan")), 3),
            "improved_fill": round(d.get("background_intensity_fill_mean", float("nan")), 3),
            "prod_empty": int(sum(f < 1e-6 for f in prod_fills)),
            "improved_empty": int(sum(f < 1e-6 for f in inten_fills)),
            "improved_max_fill": round(max(inten_fills), 3) if inten_fills else None,
        })

try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(rows).sort_values(["convention", "improved_fill"]))
except Exception:
    for r in rows:
        print(r)

print("Check: improved_empty should be 0 (continuity). improved_max_fill typically < 0.55.")


### Step C — Fixed organ dictionary + TotalSegmentator

**Goal:** one **case-independent** label map for all H&N TotalSegmentator organs + GTVp + GTVn, then run TS on QC-kept cases.

| Piece | Role |
|-------|------|
| `OrganDictionary.from_hn_canonical` | Seeds all organs up front (not discovery-order) |
| `image_processor/resources/organ_dictionary_hn_canonical.json` | Committed template |
| `label_colors` | Stable colours; **GTVp=red**, **GTVn=pink**; TS organs never use those |

Requires Step B0 (`loaded` = QC-kept cases) and Totalsegmentator installed (cell above).


In [ ]:
# --- Step C0: ensure new modules are on Colab (stale Drive/clone is common) ---
import sys
import subprocess
from pathlib import Path

assert "REPO_ROOT" in globals(), "Run setup / import-only cell first (REPO_ROOT)"
REPO_ROOT = Path(REPO_ROOT)
mod_path = REPO_ROOT / "image_processor" / "utils" / "totalsegmentator_organs.py"
print("REPO_ROOT:", REPO_ROOT)
print("expects:", mod_path, "exists=", mod_path.is_file())

if not mod_path.is_file():
    # Try git pull in clone or Drive repo
    for root in (REPO_ROOT, REPO_ROOT.parents[1] if len(REPO_ROOT.parents) > 1 else REPO_ROOT):
        if (root / ".git").exists() or (root / "psyduck-doing-phd").exists():
            print("git pull in", root)
            subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=False)
            break
    if not mod_path.is_file():
        # monorepo layout: git root may be my_tailors_drawer
        clone = Path("/content/my_tailors_drawer")
        if clone.is_dir():
            print("git pull in", clone)
            subprocess.run(["git", "-C", str(clone), "pull", "--ff-only"], check=False)

if not mod_path.is_file():
    raise FileNotFoundError(
        f"Missing {mod_path}. On Colab: open a terminal or run:\n"
        "  !git -C /content/my_tailors_drawer pull\n"
        "  # or update Drive copy at DRIVE_ROOT/repos/radcure-medical-imaging\n"
        "Then: !pip install -q -e \"{REPO_ROOT}\" && Runtime → Restart"
    )

# Reinstall editable + drop cached image_processor imports
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)])
for k in list(sys.modules):
    if k == "image_processor" or k.startswith("image_processor."):
        del sys.modules[k]

from image_processor.utils import totalsegmentator_organs as _ts
print("OK: totalsegmentator_organs from", _ts.__file__)


In [ ]:
# --- Step C1: canonical organ dictionary (case-independent) ---
from pathlib import Path
import json

from image_processor.utils.organ_dictionary import OrganDictionary
from image_processor.utils.totalsegmentator_organs import (
    DEFAULT_HN_TASKS,
    unique_hn_organ_names,
)
from image_processor.utils.label_colors import (
    rgba_by_name,
    rgba_by_index,
    COLOR_GTVP,
    COLOR_GTVN,
)

assert "REPO_ROOT" in globals() and "WORK_DIR" in globals(), (
    "Run Colab setup cells first (REPO_ROOT, WORK_DIR)"
)
REPO_ROOT = Path(REPO_ROOT)
WORK_DIR = Path(WORK_DIR)

CANONICAL_TEMPLATE = REPO_ROOT / "image_processor" / "resources" / "organ_dictionary_hn_canonical.json"
ORGAN_DICT_PATH = WORK_DIR / "audit_organ_dictionary.json"

# Prefer committed template; copy into WORK_DIR so Colab audits do not rewrite the repo file
if CANONICAL_TEMPLATE.is_file() and not ORGAN_DICT_PATH.is_file():
    ORGAN_DICT_PATH.parent.mkdir(parents=True, exist_ok=True)
    ORGAN_DICT_PATH.write_text(CANONICAL_TEMPLATE.read_text())

organ_dict = OrganDictionary.from_hn_canonical(
    str(ORGAN_DICT_PATH),
    separate_gtvp_gtvn=True,
    tasks=DEFAULT_HN_TASKS,
    save=True,
)

colors_by_name = rgba_by_name(organ_dict.dictionary)
colors_by_index = rgba_by_index(organ_dict.dictionary)

n_organs = len(unique_hn_organ_names())
assert organ_dict["background"] == 0
assert organ_dict["anatomical_region"] == 1
assert organ_dict["other-tissue"] == 2
assert "GTVp" in organ_dict and "GTVn" in organ_dict
assert colors_by_name["GTVp"] == COLOR_GTVP
assert colors_by_name["GTVn"] == COLOR_GTVN

# No TS organ may reuse tumor colours
for name, rgba in colors_by_name.items():
    if name in ("GTVp", "GTVn", "background"):
        continue
    assert rgba[:3] != COLOR_GTVP[:3], name
    assert rgba[:3] != COLOR_GTVN[:3], name

print(f"Organ dictionary: {ORGAN_DICT_PATH}")
print(f"  entries={len(organ_dict.dictionary)}  TS organs={n_organs}")
print(f"  GTVp={organ_dict['GTVp']}  GTVn={organ_dict['GTVn']}")
print(f"  tasks={list(DEFAULT_HN_TASKS)}")


In [ ]:
# --- Step C2: run TotalSegmentator on kept cases ---
from pathlib import Path
from image_processor.core.segmentator import TotalSegmentatorWrapper
from image_processor.utils.totalsegmentator_organs import DEFAULT_HN_TASKS

assert sum(len(v) for v in loaded.values()) > 0, "Run B0 first (loaded = QC-kept cases)"

try:
    import totalsegmentator  # noqa: F401
except ImportError as e:
    raise ImportError(
        "totalsegmentator not installed. Run the Step C prep pip cell, "
        "restart runtime, remount Drive, re-import."
    ) from e

segmentator = TotalSegmentatorWrapper(fast=False)
HN_TASKS = list(DEFAULT_HN_TASKS)

for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        case_dir = str(Path(d["ct_path"]).parent)
        print(f"\n=== TS {convention} {case_id} ===")
        ts_out = segmentator.run_tasks(
            case_id,
            d["ct_path"],
            case_dir,
            HN_TASKS,
        )
        d["case_dir"] = case_dir
        d["ts_output"] = ts_out
        print("→", ts_out)

print("\nTotalSegmentator done for all kept cases.")


### Step D — Combined mask (organs → other-tissue → tumor)

Merge order (same as production `MaskGenerator`):

1. Background / anatomical (prefer **Step B improved** mask)
2. TotalSegmentator organs (fixed indices from C1)
3. Leftover anatomical → **`other-tissue`**
4. Stamp **GTVp** / **GTVn** on top (separate mode)


In [ ]:
# --- Step D1: combined mask + tumor (separate GTVp/GTVn) ---
from pathlib import Path
import numpy as np

from image_processor.core.mask_generator import MaskGenerator
from image_processor.conventions import TUMOR_LABEL_MODE_SEPARATE

assert "organ_dict" in dir(), "Run C1 first"
assert any("ts_output" in d for v in loaded.values() for d in v.values()), "Run C2 first"

mask_gen = MaskGenerator(organ_dict)

for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        # Prefer improved anatomical mask from Step B; else production head_mask path
        if "background_intensity" in d:
            bg = [m.copy() for m in d["background_intensity"]]
            bg_src = "improved_B"
        else:
            bg = mask_gen.generate_background_array(d["slices"], d["ct_path"])
            bg_src = "production_head_mask"

        combined, _ = mask_gen.generate_combined_mask(
            d["slices"],
            bg,
            d["ts_output"],
        )

        tumor_path = d.get("tumor_path") or d.get("mask_path")
        assert tumor_path, f"{case_id}: missing tumor_path/mask_path"
        combined = mask_gen.update_combined_mask_with_tumor(
            tumor_path,
            d["slices"],
            combined,
            tumor_label_mode=TUMOR_LABEL_MODE_SEPARATE,
        )

        d["combined_mask"] = combined
        d["bg_source"] = bg_src

        ot = organ_dict["other-tissue"]
        gtvp, gtvn = organ_dict["GTVp"], organ_dict["GTVn"]
        fills = {
            "other-tissue": float(np.mean([np.mean(m == ot) for m in combined])),
            "GTVp": float(np.mean([np.mean(m == gtvp) for m in combined])),
            "GTVn": float(np.mean([np.mean(m == gtvn) for m in combined])),
        }
        d["combined_fills"] = fills
        print(
            f"[{convention}] {case_id}: bg={bg_src}  "
            f"other-tissue={fills['other-tissue']:.4f}  "
            f"GTVp={fills['GTVp']:.4f}  GTVn={fills['GTVn']:.4f}"
        )

print("Combined masks ready.")


In [ ]:
# --- Step D2: other-tissue summary ---
rows = []
for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        f = d.get("combined_fills", {})
        rows.append((
            convention,
            case_id,
            d.get("bg_source"),
            f.get("other-tissue"),
            f.get("GTVp"),
            f.get("GTVn"),
        ))

hdr = f"{'convention':10} {'case_id':16} {'bg':16} {'other':8} {'GTVp':8} {'GTVn':8}"
print(hdr)
print("-" * len(hdr))
ots = []
for convention, case_id, bg, ot, gp, gn in rows:
    ots.append(ot)
    print(
        f"{convention:10} {case_id:16} {str(bg):16} "
        f"{ot:8.4f} {gp:8.4f} {gn:8.4f}"
    )
if ots:
    print(f"\nMean other-tissue fill: {sum(ots)/len(ots):.4f}")


### Step E — Visualisation (CT | TS organs | + tumors)

For consecutive mid-crop slices:

1. **CT only**
2. **CT + TotalSegmentator organs** (stable colours; no red/pink)
3. **CT + organs + GTVp (red) + GTVn (pink)**

`other-tissue` is shown faintly in panel 2–3 so inflated background is visible.


In [ ]:
# --- Step E: CT vs organs vs organs+tumor ---
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from image_processor.utils.label_colors import (
    paint_label_rgba,
    organ_indices,
    COLOR_OTHER_TISSUE,
)

assert any("combined_mask" in d for v in loaded.values() for d in v.values()), "Run D1 first"

N_SLICES = 3  # consecutive around crop centre
OT_IDX = organ_dict["other-tissue"]
GTVP_IDX = organ_dict["GTVp"]
GTVN_IDX = organ_dict["GTVn"]
TS_IDXS = set(organ_indices(organ_dict.dictionary, tumors=False, specials=False))
# panel 2: TS organs + faint other-tissue
PANEL2_IDXS = TS_IDXS | {OT_IDX}
# panel 3: everything including tumors
PANEL3_IDXS = PANEL2_IDXS | {GTVP_IDX, GTVN_IDX}


def window_ct(ct2d, p1, p99):
    x = (ct2d - p1) / (p99 - p1 + 1e-8)
    return np.clip(x, 0, 1)


for convention in ("radcure", "hecktor"):
    for case_id, d in loaded[convention].items():
        slices = d["slices"]
        mid = len(slices) // 2
        half = N_SLICES // 2
        local = list(range(max(0, mid - half), min(len(slices), mid - half + N_SLICES)))
        z_list = [slices[i] for i in local]

        crop = d["ct"][:, :, slices]
        p1, p99 = np.percentile(crop, (1, 99))

        fig, axes = plt.subplots(
            N_SLICES, 3, figsize=(10, 3.2 * N_SLICES), squeeze=False
        )
        fig.suptitle(
            f"{convention} | {case_id} | bg={d.get('bg_source')} | "
            f"GTVp=red GTVn=pink",
            fontsize=11,
        )

        for row, (li, z) in enumerate(zip(local, z_list)):
            ct2d = d["ct"][:, :, z]
            lab = d["combined_mask"][li]
            base = window_ct(ct2d, p1, p99)

            # Col 0: CT
            axes[row, 0].imshow(base.T, cmap="gray", origin="lower")
            axes[row, 0].set_title(f"CT  z={z}")
            axes[row, 0].axis("off")

            # Col 1: CT + TS organs (+ other-tissue)
            ov2 = paint_label_rgba(lab, colors_by_index, include=PANEL2_IDXS)
            axes[row, 1].imshow(base.T, cmap="gray", origin="lower")
            axes[row, 1].imshow(np.transpose(ov2, (1, 0, 2)), origin="lower")
            axes[row, 1].set_title("CT + TS organs")
            axes[row, 1].axis("off")

            # Col 2: + tumors
            ov3 = paint_label_rgba(lab, colors_by_index, include=PANEL3_IDXS)
            axes[row, 2].imshow(base.T, cmap="gray", origin="lower")
            axes[row, 2].imshow(np.transpose(ov3, (1, 0, 2)), origin="lower")
            axes[row, 2].set_title("CT + organs + GTVp/GTVn")
            axes[row, 2].axis("off")

        plt.tight_layout()
        out = Path(WORK_DIR) / "figures" / f"stepE_{convention}_{case_id}.png"
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=120, bbox_inches="tight")
        plt.show()
        print("saved", out)


### Steps C–E done — checklist

- [ ] Canonical dict loaded; same GTVp/GTVn indices for every case
- [ ] TS finished (or skipped existing `total_segmentator_output/`)
- [ ] Combined masks use improved Step B background when available
- [ ] Viz: organs ≠ red/pink; GTVp red; GTVn pink
- [ ] `other-tissue` fill looks reasonable (not whole FOV / table)

**Not yet:** wire improved background into production `MaskGenerator` (separate decision).

See [`FINDINGS.md`](FINDINGS.md).
